<a href="https://colab.research.google.com/github/Kevin-March/Tesis/blob/testing/agente_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NB4 · `agente_final.ipynb` — Agente final: Neo4j + LangGraph + Mistral (+DPO) + interfaz

Es el agente del NB1 con **Mistral en el nodo `responder`**, en lugar de GPT-4o. Acá conviven las dos tecnologías de la tesis: el grafo aporta los hechos y la vigencia, y el DPO aporta la forma de explicarlos (§18).

| Celda | Qué hace | GPU |
|---|---|---|
| 0 | Instalación (correr sola y reiniciar) | — |
| 1 | GPU, Drive, credenciales, system canónico, adaptadores | sí |
| 2 | Configuración y conexiones (Neo4j, OpenAI) | — |
| 3 | Retriever híbrido del NB1 + vigencia por fecha | — |
| 4 | Consulta directa de vigencia en el grafo + verificación (60/90, 1064/97, 7599/25) | — |
| 5 | Carga Mistral base + adaptadores DPO | sí |
| 6 | Generación con Mistral, en streaming | sí |
| 7 | Agente LangGraph | — |
| 8 | Prueba en consola | sí |
| 9 | Interfaz de chat (Gradio, link público) | sí |

**Antes de empezar**
- Colab → Secrets (🔑): `OPENAI_API_KEY`, `NEO4J_URI`, `NEO4J_USERNAME`, `NEO4J_PASSWORD`, `NEO4J_DATABASE` (los mismos del NB1).
- En Drive `tesis_chatbot/`: `system_canonico.txt` y las carpetas `adaptador_dpo_ciclo1_*` que dejó el NB3.
- Entorno de ejecución: **T4 GPU**.

**Decisiones de diseño** (para el Cap. V)
1. **Solo cambia el generador.** Reescribir, graduar y reformular siguen con GPT-4o, igual que el brazo C del NB1. Así, base y DPO se diferencian únicamente en quién redacta la respuesta.
2. **Mistral recibe el mismo formato que en el entrenamiento**: system canónico + `# Contexto … # Pregunta …`, con los **2 primeros** contextos recuperados (§17.8). El historial de la conversación se usa solo para reescribir la pregunta; no entra al prompt de Mistral.
3. **Vigencia por fecha.** Una norma con `vigente_hasta` ya vencido se trata como derogada (caso 1064/97, transición hasta el 08/09/2026). Es el único cambio respecto del texto de contexto del NB1.
4. **Consulta directa de vigencia** (§17.13, caso N-14): si la pregunta nombra una norma, el agente lee su estado en el nodo del grafo y lo agrega al contexto.
5. **Generación greedy** (`do_sample=False`), igual que la celda 9 del NB3: misma pregunta, misma respuesta.
6. Para la demo y la evaluación, usar preguntas **que no estén en los tramos entrenados** (si no, se mide memoria).

## Celda 0 — Instalación
Correr **sola** y después **reiniciar la sesión** (Entorno de ejecución → Reiniciar sesión). Una vez por sesión de Colab.

In [ ]:
# ============================================================================
# NB4 · CELDA 0 — Instalación (correr SOLA, después REINICIAR la sesión)
# ============================================================================
!pip install -q "neo4j_graphrag[openai]" langgraph tiktoken tenacity bitsandbytes "gradio==6.31.0"
!pip install -q -U peft
# >>> Al terminar: Entorno de ejecución -> Reiniciar sesión, y seguir en la celda 1.
# gradio queda fijo en 6.31.0: es la versión con la que se diseñó y probó la interfaz.
# NB4 no usa Unsloth: carga Mistral con transformers + peft, que permite tener los dos
# adaptadores cargados a la vez y alternar entre ellos (y el base) sin recargar el modelo.

## Celda 1 — GPU, Drive, credenciales y adaptadores
Detecta qué adaptadores del NB3 hay en Drive. Si todavía no entrenaste ninguno, el agente corre solo con Mistral base.

In [ ]:
# ============================================================================
# NB4 · CELDA 1 — GPU, Drive, credenciales, system canónico y adaptadores
# ============================================================================
import os, torch
assert torch.cuda.is_available(), "No hay GPU. Entorno de ejecución -> Cambiar tipo -> T4."
print("GPU:", torch.cuda.get_device_name(0), "|",
      round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = "/content/drive/MyDrive/tesis_chatbot"

# Credenciales: Colab -> Secrets (icono de la llave), con los mismos nombres que en NB1.
_KEYS = ["OPENAI_API_KEY", "NEO4J_URI", "NEO4J_USERNAME", "NEO4J_PASSWORD", "NEO4J_DATABASE"]
try:
    from google.colab import userdata
    for k in _KEYS:
        os.environ[k] = userdata.get(k)
    print("Credenciales cargadas desde Colab Secrets.")
except Exception:
    import getpass
    for k in _KEYS:
        if not os.environ.get(k):
            os.environ[k] = getpass.getpass(f"{k}: ")
    print("Credenciales cargadas por teclado.")

# System canónico: se LEE del archivo que escribió NB3 (el mismo del entrenamiento, §17.12-C).
with open(f"{DRIVE_DIR}/system_canonico.txt", encoding="utf-8") as f:
    SYSTEM_CANONICO = f.read()
print("System canónico:", len(SYSTEM_CANONICO), "caracteres")

# Adaptadores DPO que dejó NB3. Se usan los que existan.
ADAPTADORES = {
    "consenso":     f"{DRIVE_DIR}/adaptador_dpo_ciclo1_consenso",
    "coincidencia": f"{DRIVE_DIR}/adaptador_dpo_ciclo1_coincidencia",
}
for nombre, ruta in list(ADAPTADORES.items()):
    ok = os.path.exists(os.path.join(ruta, "adapter_config.json"))
    print(("OK    " if ok else "FALTA ") + ruta)
    if not ok:
        del ADAPTADORES[nombre]
if not ADAPTADORES:
    print("⚠ No hay adaptadores todavía: el agente va a correr solo con Mistral base.")

## Celda 2 — Configuración y conexiones
Los mismos índices, embeddings y `TOP_K` que el NB1. `TOP_K_MISTRAL = 2` es lo que recibe Mistral, igual que en el entrenamiento DPO.

In [ ]:
# ============================================================================
# NB4 · CELDA 2 — Configuración y conexiones (mismos valores que NB1)
# ============================================================================
import logging; logging.getLogger("neo4j.notifications").setLevel(logging.ERROR)
from datetime import datetime
from zoneinfo import ZoneInfo
from neo4j import GraphDatabase
from neo4j_graphrag.embeddings import OpenAIEmbeddings
from neo4j_graphrag.llm import OpenAILLM

NEO4J_DATABASE = os.environ["NEO4J_DATABASE"]       # OJO: la base es 'ece63d51', no 'neo4j'
driver = GraphDatabase.driver(os.environ["NEO4J_URI"],
                              auth=(os.environ["NEO4J_USERNAME"], os.environ["NEO4J_PASSWORD"]))
driver.verify_connectivity()

EMBEDDING_MODEL = "text-embedding-3-small"   # el MISMO con el que se indexó (NB1)
LLM_AUX         = "gpt-4o"                   # reescribir / graduar / reformular, como en NB1
embedder = OpenAIEmbeddings(model=EMBEDDING_MODEL)
llm      = OpenAILLM(model_name=LLM_AUX, model_params={"temperature": 0})

VECTOR_INDEX_NAME         = "recuperable_embedding"
FULLTEXT_RECUPERABLE_NAME = "recuperable_texto_ft"

TOP_K             = 5      # lo que se recupera y se gradúa (igual que NB1)
TOP_K_MISTRAL     = 2      # lo que recibe Mistral: el mismo top-2 del entrenamiento DPO (§17.8)
MAX_NEW_TOKENS    = 1024
CONSULTA_VIGENCIA = True   # celda 4: leer el estado de las normas nombradas en la pregunta
FECHA_HOY = datetime.now(ZoneInfo("America/Asuncion")).date()
print("Conectado a Neo4j. Fecha de referencia para la vigencia:", FECHA_HOY.strftime("%d/%m/%Y"))

## Celda 3 — Retriever híbrido del NB1 + vigencia por fecha
Al final imprime el encabezado de lo que trae para la Ley 60/90: el primero tiene que decir `[DEROGADA por 7548/25]`.

In [ ]:
# ============================================================================
# NB4 · CELDA 3 — Retriever GraphRAG híbrido del NB1 + vigencia por fecha
# ============================================================================
# Es el retriever del NB1 (vector + fulltext + expansión por el grafo, con escapado
# de Lucene). Dos agregados, ninguno cambia QUÉ se recupera:
#   1. La consulta Cypher devuelve además el TIPO de la norma que deroga (para que la
#      interfaz diga "derogada por la Ley 7548/25" y no solo "7548/25").
#   2. VIGENCIA POR FECHA: si vigente_hasta ya pasó, la norma se marca como DEROGADA.
#      En NB1 la marca era siempre "[vigente hasta <fecha>]", aunque la fecha ya hubiera
#      vencido (caso 1064/97: transición hasta el 08/09/2026).
# El texto de contexto que ve el modelo es el mismo del NB1 en todo lo demás.
# ============================================================================
from datetime import date
from typing import Optional
from neo4j_graphrag.retrievers import HybridCypherRetriever
from neo4j_graphrag.types import RetrieverResultItem

_TIPO_DE = "head([l IN labels({v}) WHERE l IN ['Ley','Decreto','Resolucion']])"

RETRIEVAL_QUERY = f"""
// --- Rama A: la semilla es un Articulo -> su norma padre ---
OPTIONAL MATCH (norma)-[:TIENE_ARTICULO]->(node)
    WHERE norma:Ley OR norma:Decreto OR norma:Resolucion
WITH node, collect(norma)[0] AS norma

// --- Rama B: la semilla es una Ficha -> las normas que describe, con vigencia ---
OPTIONAL MATCH (node)-[:DESCRIBE]->(d)
    WHERE d:Ley OR d:Decreto OR d:Resolucion
WITH node, norma, collect(CASE WHEN d IS NULL THEN null ELSE {{
    numero:           toString(d.numero),
    tipo:             {_TIPO_DE.format(v='d')},
    estado:           coalesce(d.estado, 'sin_dato'),
    derogada_por:     d.derogada_por,
    vigente_hasta:    d.vigente_hasta,
    derogada_por_rel: [ (d)<-[:DEROGA]-(x) | toString(x.numero) ],
    derogada_por_ui:  [ (d)<-[:DEROGA]-(x) | coalesce({_TIPO_DE.format(v='x')}, '') + ' ' + toString(x.numero) ]
}} END) AS descritas_raw
WITH node, norma, [x IN descritas_raw WHERE x IS NOT NULL] AS descritas

RETURN
    CASE WHEN node:Ficha THEN 'ficha' ELSE 'articulo' END AS tipo_nodo,
    node.texto  AS texto,
    node.titulo AS ficha_titulo,
    node.estado AS ficha_estado,
    node.fuente AS ficha_fuente,
    node.nota   AS ficha_nota,
    descritas   AS descritas,

    // --- campos de articulo (null si la semilla es una Ficha) ---
    node.numero     AS articulo,
    node.ley_numero AS norma_numero,
    CASE WHEN norma IS NULL THEN null ELSE {_TIPO_DE.format(v='norma')} END AS norma_tipo,
    CASE WHEN norma IS NULL THEN null ELSE norma.nombre_completo END AS norma_titulo,
    CASE WHEN norma IS NULL THEN 'sin_dato' ELSE coalesce(norma.estado,'sin_dato') END AS estado,
    CASE WHEN norma IS NULL THEN null ELSE norma.derogada_por END AS derogada_por,
    CASE WHEN norma IS NULL THEN null ELSE norma.vigente_hasta END AS vigente_hasta,
    CASE WHEN norma IS NULL THEN null ELSE norma.fuente_url END AS fuente_url,
    CASE WHEN norma IS NULL THEN [] ELSE [ (norma)<-[:DEROGA]-(x)     | toString(x.numero) ] END AS derogada_por_rel,
    CASE WHEN norma IS NULL THEN [] ELSE [ (norma)<-[:DEROGA]-(x)     | coalesce({_TIPO_DE.format(v='x')}, '') + ' ' + toString(x.numero) ] END AS derogada_por_ui,
    CASE WHEN norma IS NULL THEN [] ELSE [ (norma)<-[:MODIFICA]-(y)   | toString(y.numero) ] END AS modificada_por_rel,
    CASE WHEN norma IS NULL THEN [] ELSE [ (norma)<-[:REGLAMENTA]-(dd) | toString(dd.numero) + ' — ' + coalesce(dd.nombre_completo,'') ] END AS reglamentada_por,
    CASE WHEN norma IS NULL THEN [] ELSE [ (norma)-[:REGLAMENTA]->(l) | toString(l.numero) + ' — ' + coalesce(l.nombre_completo,'') ] END AS reglamenta_a
"""


def _fecha(v):
    """vigente_hasta puede venir como Date de Neo4j o como texto 'AAAA-MM-DD'."""
    if v is None:
        return None
    try:
        return date.fromisoformat(str(v)[:10])
    except ValueError:
        return None


def vigencia(estado, derogada_por=None, vigente_hasta=None, derogada_por_rel=()):
    """Estado EFECTIVO de una norma a FECHA_HOY.
    clave: vigente | vigente_hasta | derogada | sin_dato | otro"""
    hasta = _fecha(vigente_hasta)
    por = derogada_por or next((x for x in (derogada_por_rel or []) if x and str(x).strip()), None)
    if hasta and hasta < FECHA_HOY:            # derogación diferida que ya se cumplió (1064/97)
        return {"clave": "derogada", "por": por, "hasta": hasta, "estado": estado}
    if hasta:                                  # todavía en período de transición
        return {"clave": "vigente_hasta", "por": por, "hasta": hasta, "estado": estado}
    e = str(estado or "sin_dato").lower()
    if derogada_por or e == "derogada":
        return {"clave": "derogada", "por": por, "hasta": None, "estado": estado}
    return {"clave": e if e in ("vigente", "sin_dato") else "otro", "por": None, "hasta": None, "estado": estado}


def _marca_vigencia(v, vigente_hasta_raw=None):
    """Marca que ve el modelo. Para los casos que ya existían, el MISMO texto que NB1."""
    if v["clave"] == "derogada" and v["hasta"]:
        m = f"DEROGADA: rigió hasta el {v['hasta']:%d/%m/%Y}"
        return f"[{m}; reemplazada por {v['por']}]" if v["por"] else f"[{m}]"
    if v["clave"] == "derogada":
        return f"[DEROGADA por {v['por']}]" if v["por"] else "[estado: derogada]"
    if v["clave"] == "vigente_hasta":
        return f"[vigente hasta {vigente_hasta_raw}]"
    if v["clave"] == "otro":
        return f"[estado: {v['estado']}]"
    return ""


def _entrada(tipo, numero, articulo, v, por_ui=None):
    """Una fila de la lista de normas que muestra la interfaz."""
    return {"tipo": tipo or "", "numero": str(numero or ""), "articulo": articulo,
            "clave": v["clave"], "por": por_ui or v["por"], "hasta": v["hasta"], "estado": v["estado"]}


def _primero(lista):
    return next((x.strip() for x in (lista or []) if x and x.strip()), None)


def formatear(record):
    # =========================== FICHA ===========================
    if record.get("tipo_nodo") == "ficha":
        titulo = record.get("ficha_titulo") or "Ficha"
        fuente = record.get("ficha_fuente") or "MIC"
        encabezado = f"[FICHA {fuente}] {titulo}"
        if record.get("ficha_estado") == "historica":
            encabezado += "  [DOCUMENTO HISTÓRICO: describe un régimen DEROGADO]"

        lineas, obsoletas, normas = [], [], []
        for d in (record.get("descritas") or []):
            v = vigencia(d.get("estado"), d.get("derogada_por"), d.get("vigente_hasta"), d.get("derogada_por_rel"))
            marca = _marca_vigencia(v, d.get("vigente_hasta"))
            lineas.append(f"{d.get('numero')} {marca}".strip())
            if v["clave"] in ("derogada", "vigente_hasta"):
                obsoletas.append(f"{d.get('numero')} {marca}".strip())
            normas.append(_entrada(d.get("tipo"), d.get("numero"), None, v, _primero(d.get("derogada_por_ui"))))

        pie = ""
        if lineas:
            pie += "\n\nNormas que describe esta ficha: " + " · ".join(lineas)
        if obsoletas:
            pie += ("\n⚠ ADVERTENCIA: esta ficha hace referencia a normas que YA NO ESTÁN "
                    "PLENAMENTE VIGENTES (" + " · ".join(obsoletas) + "). La ficha puede estar "
                    "desactualizada en esos puntos: no presentes esa información como vigente.")

        return RetrieverResultItem(
            content=f"{encabezado}\n{record.get('texto') or ''}{pie}",
            metadata={"tipo": "ficha", "titulo": titulo, "estado": record.get("ficha_estado"),
                      "describe": [d.get("numero") for d in (record.get("descritas") or [])],
                      "normas": normas},
        )

    # ========================== ARTÍCULO ==========================
    v = vigencia(record.get("estado"), record.get("derogada_por"), record.get("vigente_hasta"),
                 record.get("derogada_por_rel"))
    marca = _marca_vigencia(v, record.get("vigente_hasta"))
    marca = f"  {marca}" if marca else ""
    tipo = record.get("norma_tipo") or ""
    num = record.get("norma_numero") or ""
    encabezado = f"{tipo} {num}, Artículo {record.get('articulo')}{marca}".strip()

    extras = []
    if record.get("reglamentada_por"):
        extras.append("Reglamentada por: " + "; ".join(record["reglamentada_por"]))
    if record.get("reglamenta_a"):
        extras.append("Reglamenta a: " + "; ".join(record["reglamenta_a"]))
    if record.get("modificada_por_rel"):
        extras.append("Modificada por: " + ", ".join(record["modificada_por_rel"]))
    extra_txt = ("\n" + " | ".join(extras)) if extras else ""

    return RetrieverResultItem(
        content=f"{encabezado}\n{record.get('texto') or ''}{extra_txt}",
        metadata={"tipo": "articulo", "norma": num, "articulo": record.get("articulo"),
                  "estado": record.get("estado"), "derogada_por": record.get("derogada_por"),
                  "fuente_url": record.get("fuente_url"),
                  "normas": [_entrada(tipo, num, record.get("articulo"), v, _primero(record.get("derogada_por_ui")))]},
    )


# ---- Escapado de Lucene (idéntico a NB1: sin esto, "60/90" rompe el fulltext) ----
LUCENE_ESPECIALES = set('\\+-&|!(){}[]^"~*?:/')

def escapar_lucene(texto: str) -> str:
    if not texto:
        return texto
    return "".join(("\\" + ch) if ch in LUCENE_ESPECIALES else ch for ch in texto)


class HybridCypherRetrieverSeguro(HybridCypherRetriever):
    """HybridCypherRetriever que no se rompe con '/' ni '?' en la consulta (NB1)."""
    def get_search_results(self, query_text: str, query_vector: Optional[list] = None, **kwargs):
        if query_vector is None and query_text:
            query_vector = self.embedder.embed_query(query_text)      # vector: texto ORIGINAL
        return super().get_search_results(query_text=escapar_lucene(query_text),   # Lucene: ESCAPADO
                                          query_vector=query_vector, **kwargs)


# --- instanciar y probar ---------------------------------------------------
retriever = HybridCypherRetrieverSeguro(
    driver=driver,
    vector_index_name=VECTOR_INDEX_NAME,
    fulltext_index_name=FULLTEXT_RECUPERABLE_NAME,
    retrieval_query=RETRIEVAL_QUERY,
    embedder=embedder,
    result_formatter=formatear,
    neo4j_database=NEO4J_DATABASE,
)
for i, it in enumerate(retriever.search(query_text="¿Qué incentivos fiscales ofrece la Ley 60/90?", top_k=TOP_K).items, 1):
    print(f"[{i}]", it.content.splitlines()[0][:120])

## Celda 4 — Vigencia directa en el grafo
Mirá la salida: **1064/97 tiene que aparecer como DEROGADA** ("rigió hasta el 08/09/2026"), 60/90 derogada y 7599/25 vigente.

In [ ]:
# ============================================================================
# NB4 · CELDA 4 — Consulta directa de vigencia en el grafo (+ verificación)
# ============================================================================
# §17.13 (N-14): "¿La Ley 7599/25 está vigente?" fallaba porque la vigencia no vive
# en el TEXTO de los artículos sino en el NODO de la norma (estado, derogada_por,
# vigente_hasta, relaciones DEROGA). Si la pregunta nombra una norma, el agente la
# busca por número y agrega una línea "[VIGENCIA EN EL GRAFO]" al contexto.
# ============================================================================
import re

_RE_NORMA  = re.compile(r"(?i)\b(?:ley|decreto|resoluci[oó]n)\s*(?:n[º°o]?\.?\s*)?(\d[\d.]*(?:/\d{2,4})?)")
_RE_NUMERO = re.compile(r"(?<![\d.])(\d{1,5}/\d{2,4})\b")      # "la 60/90", sin la palabra "Ley"

CYPHER_VIGENCIA = f"""
MATCH (n) WHERE (n:Ley OR n:Decreto OR n:Resolucion) AND toString(n.numero) IN $nums
RETURN {_TIPO_DE.format(v='n')} AS tipo, toString(n.numero) AS numero,
       n.nombre_completo AS titulo, coalesce(n.estado, 'sin_dato') AS estado,
       n.derogada_por AS derogada_por, n.vigente_hasta AS vigente_hasta,
       [ (n)<-[:DEROGA]-(x) | toString(x.numero) ] AS derogada_por_rel,
       [ (n)<-[:DEROGA]-(x) | coalesce({_TIPO_DE.format(v='x')}, '') + ' ' + toString(x.numero) ] AS derogada_por_ui
"""


def _variantes(ref):
    """En el grafo conviven '60/90', '6480' (sin año) y '11771' (sin punto)."""
    sin_punto = ref.replace(".", "")
    orden = [ref, sin_punto]
    if "/" in sin_punto:
        numero, anio = sin_punto.split("/", 1)
        if len(anio) == 4:
            orden.append(f"{numero}/{anio[2:]}")     # 6480/2020 -> 6480/20
        orden.append(numero)                          # 6480/20   -> 6480
    return list(dict.fromkeys(orden))                 # de la más específica a la más general


def vigencia_en_grafo(consulta):
    refs = list(dict.fromkeys(_RE_NORMA.findall(consulta) + _RE_NUMERO.findall(consulta)))
    refs = [r.rstrip(".") for r in refs if r.rstrip(".")]
    if not refs:
        return []
    variantes = {r: _variantes(r) for r in refs}
    nums = sorted({v for vs in variantes.values() for v in vs})
    recs, _, _ = driver.execute_query(CYPHER_VIGENCIA, nums=nums, database_=NEO4J_DATABASE)
    por_numero = {r["numero"]: r.data() for r in recs}

    items, vistos = [], set()
    for r in refs:
        d = next((por_numero[v] for v in variantes[r] if v in por_numero), None)
        if d is None or d["numero"] in vistos:
            continue
        vistos.add(d["numero"])
        v = vigencia(d["estado"], d["derogada_por"], d["vigente_hasta"], d["derogada_por_rel"])
        marca = _marca_vigencia(v, d["vigente_hasta"]) or (
            "[vigente]" if v["clave"] == "vigente" else "[sin dato de vigencia en el grafo]")
        titulo = f" — {d['titulo']}" if d.get("titulo") else ""
        items.append({
            "content": f"[VIGENCIA EN EL GRAFO] {d['tipo']} {d['numero']}{titulo} {marca}",
            "metadata": {"tipo": "vigencia",
                         "normas": [_entrada(d["tipo"], d["numero"], None, v, _primero(d["derogada_por_ui"]))]},
        })
    return items


# --- verificación (mirar una vez) ------------------------------------------
recs, _, _ = driver.execute_query(
    "MATCH (n:Ley) WHERE toString(n.numero) = '1064/97' "
    "RETURN n.estado AS estado, n.derogada_por AS derogada_por, n.vigente_hasta AS vigente_hasta, "
    "[ (n)<-[:DEROGA]-(x) | toString(x.numero) ] AS deroga_rel", database_=NEO4J_DATABASE)
print("1064/97 en el grafo:", recs[0].data() if recs else "NO ESTÁ")
for q in ["¿Qué incentivos ofrece la Ley 60/90?",
          "¿La Ley 1064/97 de maquila sigue vigente?",
          "¿La Ley 7599/25 está vigente?"]:
    print("\n" + q)
    for it in vigencia_en_grafo(q):
        print("   ", it["content"])
print("\nEsperado: 60/90 DEROGADA · 1064/97 DEROGADA ('rigió hasta el 08/09/2026') · 7599/25 vigente")

## Celda 5 — Mistral base + adaptadores (GPU)
La primera vez descarga el modelo (~4 GB) y tarda unos minutos. Al final muestra los modelos disponibles y la VRAM usada.

In [ ]:
# ============================================================================
# NB4 · CELDA 5 — Mistral base + adaptadores DPO (GPU)
# ============================================================================
# Mismo modelo base que NB3. Los adaptadores se cargan TODOS sobre el mismo base:
#   - "base"          -> con el adaptador apagado (model.disable_adapter())
#   - "consenso"      -> model.set_adapter("consenso")
#   - "coincidencia"  -> model.set_adapter("coincidencia")
# Así se cambia de modelo en la interfaz sin recargar nada (~5 GB de VRAM en total).
# ============================================================================
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

BASE_MODELO = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit"     # el mismo base del NB3

# El tokenizer sale de la carpeta del adaptador: es exactamente el del entrenamiento.
tokenizer = AutoTokenizer.from_pretrained(next(iter(ADAPTADORES.values()), BASE_MODELO))

# El checkpoint ya viene en 4 bits; esto solo fija el cómputo en float16 (la T4 no tiene bfloat16).
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
base = AutoModelForCausalLM.from_pretrained(BASE_MODELO, quantization_config=bnb,
                                            dtype=torch.float16, device_map={"": 0})

if ADAPTADORES:
    nombres = list(ADAPTADORES)
    model = PeftModel.from_pretrained(base, ADAPTADORES[nombres[0]], adapter_name=nombres[0])
    for n in nombres[1:]:
        model.load_adapter(ADAPTADORES[n], adapter_name=n)
else:
    model = base
model.eval()

# Etiqueta que ve el usuario -> clave interna. El primero es el que arranca seleccionado.
MODELOS = {f"Mistral + DPO ({n})": n for n in ADAPTADORES}
MODELOS["Mistral base"] = "base"
print("Modelos disponibles:", list(MODELOS))
print(f"VRAM usada: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

## Celda 6 — Generación con Mistral, en streaming
Imprime el prompt que recibe Mistral: tiene que tener el system canónico y el formato `# Contexto … # Pregunta …`.

In [ ]:
# ============================================================================
# NB4 · CELDA 6 — Generación con Mistral, en streaming (GPU)
# ============================================================================
# El prompt es EXACTAMENTE el del entrenamiento DPO (NB3, celdas 2 y 8):
#   system = system_canonico.txt
#   user   = "# Contexto:\n{contexto}\n\n# Pregunta:\n{pregunta}"
# Greedy (do_sample=False), igual que la celda 9 del NB3.
# ============================================================================
from threading import Thread
from contextlib import nullcontext
from transformers import TextIteratorStreamer


def prompt_mistral(contexto, pregunta):
    msgs = [{"role": "system", "content": SYSTEM_CANONICO},
            {"role": "user", "content": f"# Contexto:\n{contexto}\n\n# Pregunta:\n{pregunta}"}]
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)


def _modelo_activo(clave):
    """Prende el adaptador pedido, o los apaga todos para el base."""
    if clave == "base":
        return model.disable_adapter() if hasattr(model, "disable_adapter") else nullcontext()
    model.set_adapter(clave)
    return nullcontext()


def generar_stream(contexto, pregunta, clave_modelo):
    """Va devolviendo el texto a medida que Mistral lo genera."""
    ins = tokenizer(prompt_mistral(contexto, pregunta), return_tensors="pt").to(model.device)
    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True, timeout=300)
    errores = []

    def _correr():
        try:
            model.generate(**ins, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                           pad_token_id=tokenizer.eos_token_id, streamer=streamer)
        except Exception as e:          # sin esto, un error en el hilo deja colgado al streamer
            errores.append(e)
            streamer.end()

    with _modelo_activo(clave_modelo):
        hilo = Thread(target=_correr)
        hilo.start()
        for trozo in streamer:
            yield trozo
        hilo.join()
    if errores:
        raise errores[0]


# Control rápido: así se ve el prompt que recibe Mistral.
print(prompt_mistral("<contexto>", "<pregunta>"))

## Celda 7 — Agente LangGraph
El grafo del NB1 sin cambios en su forma: solo `responder` usa Mistral.

In [ ]:
# ============================================================================
# NB4 · CELDA 7 — Agente LangGraph (el del NB1, con Mistral en `responder`)
# ============================================================================
# reescribir -> recuperar -> graduar -> { responder | reformular↻ | sin_contexto }
#   reescribir / graduar / reformular: GPT-4o, mismos prompts que NB1.
#   recuperar: retriever híbrido (top-5 para graduar) + vigencia directa del grafo.
#   responder: MISTRAL (base o DPO) con el top-2 + la vigencia, en streaming.
# Sin checkpointer: el historial lo manda la interfaz en cada consulta, así una
# conversación retomada desde el historial del navegador sigue teniendo memoria.
# ============================================================================
from typing import Annotated, TypedDict
import operator
from langgraph.graph import StateGraph, START, END
from langgraph.config import get_stream_writer


class EstadoConv(TypedDict):
    messages: Annotated[list, operator.add]
    modelo: str        # "base" | "consenso" | "coincidencia"
    consulta: str      # consulta (reescrita) con la que se busca
    context: str       # contexto para graduar (top-5 + vigencia)
    items: list        # lo que recibe Mistral -> también arma la lista de normas de la interfaz
    intentos: int
    relevante: bool


def reescribir(state: EstadoConv):
    if len(state["messages"]) <= 1:
        return {"consulta": state["messages"][-1]["content"], "intentos": 0}
    historial = "\n".join(f'{m["role"]}: {m["content"]}' for m in state["messages"][:-1])
    pregunta = state["messages"][-1]["content"]
    prompt = (
        "Dada la conversación previa y una pregunta de seguimiento, reescribí la pregunta como "
        "una consulta AUTÓNOMA que se entienda sin el historial (resolvé 'esa ley', 'ese régimen'). "
        "Si ya es autónoma, devolvela igual. Devolvé SOLO la consulta, sin comillas.\n\n"
        f"# Conversación:\n{historial}\n\n# Pregunta:\n{pregunta}\n\n# Consulta autónoma:"
    )
    return {"consulta": llm.invoke(prompt).content.strip(), "intentos": 0}


def recuperar(state: EstadoConv):
    res = retriever.search(query_text=state["consulta"], top_k=TOP_K)
    items = [{"content": it.content, "metadata": it.metadata or {}} for it in res.items]
    vig = vigencia_en_grafo(state["consulta"]) if CONSULTA_VIGENCIA else []
    return {"context": "\n\n".join(x["content"] for x in vig + items),
            "items": vig + items[:TOP_K_MISTRAL],
            "intentos": state.get("intentos", 0) + 1}


def graduar(state: EstadoConv):
    prompt = (
        "Decidí si el CONTEXTO alcanza para responder la PREGUNTA de forma fundamentada. "
        "Respondé SOLO con 'SI' o 'NO'.\n\n"
        "IMPORTANTE: si el contexto contiene artículos o fichas sobre la norma consultada, "
        "alcanza — AUNQUE la norma figure como DEROGADA, modificada o no vigente. Advertir que "
        "una norma está derogada (y por cuál fue reemplazada) es una respuesta VÁLIDA y útil, "
        "no una falta de información. Respondé 'NO' solo si el contexto es de otra materia.\n\n"
        f"# Pregunta:\n{state['consulta']}\n\n# Contexto:\n{state['context'][:4000]}\n\n# ¿Alcanza? (SI/NO):"
    )
    return {"relevante": llm.invoke(prompt).content.strip().upper().startswith("SI")}


def decidir(state: EstadoConv):
    if state["relevante"]:
        return "responder"
    if state["intentos"] < 2:
        return "reformular"
    return "sin_contexto"


def reformular(state: EstadoConv):
    prompt = (
        "La búsqueda anterior no trajo contexto suficiente. Reformulá la consulta con otros "
        "términos o de forma más general. Devolvé SOLO la nueva consulta.\n\n"
        f"# Consulta anterior:\n{state['consulta']}"
    )
    return {"consulta": llm.invoke(prompt).content.strip()}


def responder(state: EstadoConv):
    escribir = get_stream_writer()
    contexto = "\n\n".join(x["content"] for x in state["items"])
    texto = ""
    for trozo in generar_stream(contexto, state["consulta"], state["modelo"]):
        texto += trozo
        escribir({"token": trozo})
    return {"messages": [{"role": "assistant", "content": texto.strip()}]}


def sin_contexto(state: EstadoConv):
    return {"items": [], "messages": [{"role": "assistant", "content":
        "No encontré en la base normativa cargada información suficiente para responder eso "
        "con fundamento. ¿Podés reformular la pregunta o dar más detalle?"}]}


builder = StateGraph(EstadoConv)
for nombre, fn in [("reescribir", reescribir), ("recuperar", recuperar), ("graduar", graduar),
                   ("reformular", reformular), ("responder", responder), ("sin_contexto", sin_contexto)]:
    builder.add_node(nombre, fn)
builder.add_edge(START, "reescribir")
builder.add_edge("reescribir", "recuperar")
builder.add_edge("recuperar", "graduar")
builder.add_conditional_edges("graduar", decidir, {
    "responder": "responder", "reformular": "reformular", "sin_contexto": "sin_contexto"})
builder.add_edge("reformular", "recuperar")       # bucle acotado por intentos < 2
builder.add_edge("responder", END)
builder.add_edge("sin_contexto", END)
chat_graph = builder.compile()


def consultar(pregunta, historial=(), modelo="base"):
    """Corre el agente. Devuelve ("token", texto) mientras Mistral escribe y ("fin", estado) al final."""
    msgs = [{"role": m["role"], "content": m["content"]} for m in historial]
    msgs.append({"role": "user", "content": pregunta})
    final = None
    for modo, dato in chat_graph.stream({"messages": msgs, "modelo": modelo},
                                        stream_mode=["custom", "values"]):
        if modo == "custom" and "token" in dato:
            yield "token", dato["token"]
        elif modo == "values":
            final = dato
    yield "fin", final


# ---- Lista de normas de una respuesta (la usan la consola y la interfaz) ----
_TIPO_UI = {"Resolucion": "Resolución"}
_ARTICULO_ES = {"Ley": "la", "Decreto": "el", "Resolucion": "la", "Resolución": "la"}

def _con_articulo(ref):
    """'Ley 7548/25' -> 'la Ley 7548/25'; un número suelto queda igual."""
    ref = (ref or "").strip()
    primera = ref.split(" ")[0]
    if primera in _ARTICULO_ES:
        return f"{_ARTICULO_ES[primera]} {_TIPO_UI.get(primera, primera)}{ref[len(primera):]}"
    return ref


def normas_de(items):
    """Filas sin repetir. Si hay un artículo de una norma, no se repite la norma sola."""
    filas = []
    for it in items:
        md = it["metadata"]
        if md.get("tipo") == "ficha":
            filas.append({"clave": "ficha", "titulo": md.get("titulo") or "Ficha del MIC"})
        filas.extend(n for n in md.get("normas", []) if n.get("numero"))
    con_articulo = {(n["tipo"], n["numero"]) for n in filas if n.get("articulo") not in (None, "")}
    vistas, out = set(), []
    for n in filas:
        if n["clave"] == "ficha":
            clave = ("ficha", n["titulo"])
        else:
            if n.get("articulo") in (None, "") and (n["tipo"], n["numero"]) in con_articulo:
                continue
            clave = (n["tipo"], n["numero"], n.get("articulo"))
        if clave not in vistas:
            vistas.add(clave)
            out.append(n)
    return out


def etiqueta_norma(n):
    """(nombre, estado) en lenguaje claro: ('Ley 60/90, art. 1', 'derogada por la Ley 7548/25')."""
    if n["clave"] == "ficha":
        return n["titulo"], "ficha del MIC"
    nombre = f"{_TIPO_UI.get(n['tipo'], n['tipo'])} {n['numero']}".strip()
    if n.get("articulo") not in (None, ""):
        nombre += f", art. {n['articulo']}"
    c, por, hasta = n["clave"], n.get("por"), n.get("hasta")
    if c == "derogada" and hasta:
        estado = f"rigió hasta el {hasta:%d/%m/%Y}" + (f", reemplazada por {_con_articulo(por)}" if por else "")
    elif c == "derogada":
        estado = f"derogada por {_con_articulo(por)}" if por else "derogada"
    elif c == "vigente_hasta":
        estado = f"vigente hasta el {hasta:%d/%m/%Y}"
    elif c == "vigente":
        estado = "vigente"
    elif c == "sin_dato":
        estado = "sin dato de vigencia"
    else:
        estado = str(n.get("estado") or "estado no informado")
    return nombre, estado


print("Agente listo.")

## Celda 8 — Prueba en consola
Antes de abrir la interfaz. La misma pregunta con el base y con el DPO, una repregunta, la 1064/97 y una pregunta fuera del dominio.

In [ ]:
# ============================================================================
# NB4 · CELDA 8 — Prueba en consola (antes de abrir la interfaz)
# ============================================================================
# Cinco casos: derogación (base y DPO), repregunta, derogación diferida vencida
# y pregunta fuera del dominio. Ninguna de estas preguntas está en los tramos entrenados.
# ============================================================================
def probar(pregunta, modelo="base", historial=()):
    print("=" * 80 + f"\n[{modelo}] {pregunta}\n")
    final, hubo_texto = None, False
    for tipo, dato in consultar(pregunta, historial, modelo):
        if tipo == "token":
            print(dato, end="", flush=True)
            hubo_texto = True
        else:
            final = dato
    if not hubo_texto:
        print(final["messages"][-1]["content"])
    if final["consulta"] != pregunta:
        print(f"\n\n  consulta reescrita: {final['consulta']}")
    print(f"\n  búsquedas: {final['intentos']}")
    for n in normas_de(final.get("items") or []):
        nombre, estado = etiqueta_norma(n)
        print(f"   · {nombre} — {estado}")
    return final


DPO = next((n for n in ADAPTADORES), "base")      # el primer adaptador disponible

r = probar("¿Qué incentivos fiscales ofrece la Ley 60/90?", "base")
probar("¿Qué incentivos fiscales ofrece la Ley 60/90?", DPO)
probar("¿Y si mi proyecto ya estaba aprobado bajo esa ley?", DPO,
       historial=[{"role": "user", "content": "¿Qué incentivos fiscales ofrece la Ley 60/90?"},
                  {"role": "assistant", "content": r["messages"][-1]["content"]}])
probar("¿La Ley 1064/97 de maquila sigue vigente?", DPO)
probar("¿Cuál es la mejor receta de sopa paraguaya?", DPO)    # fuera de dominio: no llama a Mistral

## Celda 9 — Interfaz de chat
Abrí el link `https://….gradio.live` que aparece abajo. Funciona mientras esta sesión de Colab esté abierta.

Para una **evaluación ciega**, poné `MOSTRAR_MODELO = False` y `MOSTRAR_SELECTOR = False` arriba de la celda.

In [ ]:
# ============================================================================
# NB4 · CELDA 9 — Interfaz de chat (Gradio, link público)
# ============================================================================
# share=True genera un link público (como ngrok en ALIBot, pero sin túnel externo).
# El link funciona mientras esta sesión de Colab esté abierta (máximo una semana).
# El historial de conversaciones se guarda en el navegador de cada persona.
#
# Para una evaluación CIEGA con evaluadores: MOSTRAR_MODELO = False y
# MOSTRAR_SELECTOR = False (responde siempre el primer modelo de MODELOS).
# ============================================================================
import html, re, traceback
import gradio as gr

MOSTRAR_MODELO   = True     # pie "Respondió Mistral + DPO (consenso)" debajo de cada respuesta
MOSTRAR_SELECTOR = True     # selector de modelo arriba del chat
SUGERENCIAS = ["¿Qué incentivos fiscales ofrece la Ley 60/90?",      # ninguna está en los tramos
               "¿La Ley 1064/97 de maquila sigue vigente?",
               "¿Cómo constituyo una EAS?"]


# ---- Del historial de Gradio al formato del agente ----------------------------
def _texto(content):
    if isinstance(content, str):
        return content
    if isinstance(content, list):                    # Gradio 6: [{"type": "text", "text": ...}]
        return "".join(b.get("text", "") for b in content if isinstance(b, dict))
    return str(content or "")


def _limpiar(texto):
    """Saca de una respuesta la lista de normas y el aviso de búsqueda (son de la interfaz)."""
    return re.split(r'<div class="normas"|<p class="buscando"', texto)[0].strip()


def _historial(history):
    out = []
    for m in history or []:
        t = _limpiar(_texto(m.get("content")))
        if m.get("role") in ("user", "assistant") and t:
            out.append({"role": m["role"], "content": t})
    return out


# ---- Lista de normas debajo de cada respuesta ---------------------------------
def bloque_normas(normas, etiqueta_modelo=None):
    filas = []
    for n in normas:
        nombre, estado = (html.escape(x) for x in etiqueta_norma(n))
        if n["clave"] == "derogada":
            filas.append(f'<li class="norma norma-derogada"><del>{nombre}</del><span class="estado">{estado}</span></li>')
        else:
            clase = "norma-vigente" if n["clave"] in ("vigente", "vigente_hasta") else "norma-neutra"
            filas.append(f'<li class="norma {clase}"><span class="nombre">{nombre}</span><span class="estado">{estado}</span></li>')
    pie = f'<p class="modelo-usado">Respondió {html.escape(etiqueta_modelo)}</p>' if etiqueta_modelo else ""
    if not filas:
        return f'<div class="normas sin-normas">{pie}</div>' if pie else ""
    return ('<div class="normas"><p class="normas-titulo">Normas en las que se basa esta respuesta</p>'
            f'<ul>{"".join(filas)}</ul>{pie}</div>')


def responder_ui(mensaje, history, etiqueta_modelo):
    clave = MODELOS.get(etiqueta_modelo, next(iter(MODELOS.values())))
    yield '<p class="buscando">Buscando en las normas…</p>'
    texto, final = "", None
    try:
        for tipo, dato in consultar(mensaje, _historial(history), clave):
            if tipo == "token":
                texto += dato
                yield texto
            else:
                final = dato
    except Exception as e:
        traceback.print_exc()
        yield f"No pude completar la consulta ({type(e).__name__}). Probá de nuevo en unos segundos."
        return
    respuesta = final["messages"][-1]["content"] if final else texto
    bloque = bloque_normas(normas_de(final.get("items") or []), etiqueta_modelo if MOSTRAR_MODELO else None)
    yield respuesta + ("\n\n" + bloque if bloque else "")


# ---- Diseño -------------------------------------------------------------------
HEAD = """
<link rel="preconnect" href="https://fonts.googleapis.com">
<link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
<link href="https://fonts.googleapis.com/css2?family=Newsreader:opsz,wght@6..72,400;6..72,500;6..72,600&family=Public+Sans:wght@400;600;700&display=swap" rel="stylesheet">
<script>
// Gradio no traduce "New chat": se cambia el texto del botón.
new MutationObserver(() => {
  document.querySelectorAll('button').forEach(b => b.childNodes.forEach(n => {
    if (n.nodeType === 3 && n.textContent.trim() === 'New chat') n.textContent = 'Nueva consulta';
  }));
}).observe(document, {subtree: true, childList: true});
// Las sugerencias del estado vacío escriben la consulta y la envían.
document.addEventListener('click', e => {
  const a = e.target.closest('.sugerencia'); if (!a) return;
  e.preventDefault();
  const ta = document.querySelector('#entrada textarea');
  Object.getOwnPropertyDescriptor(HTMLTextAreaElement.prototype, 'value').set.call(ta, a.textContent);
  ta.dispatchEvent(new Event('input', {bubbles: true}));
  setTimeout(() => document.querySelector('#entrada .submit-button').click(), 60);
});
// Modo claro / oscuro: Gradio pone la clase "dark" en <body> según el sistema;
// el botón la cambia y la elección queda guardada en este navegador.
function rotularModo() {
  const b = document.querySelector('.modo');
  if (b) b.textContent = document.body.classList.contains('dark') ? 'Modo claro' : 'Modo oscuro';
}
function aplicarModoGuardado() {
  let m = null; try { m = localStorage.getItem('modo'); } catch (e) {}
  if (m === 'oscuro') document.body.classList.add('dark');
  if (m === 'claro') document.body.classList.remove('dark');
  rotularModo();
}
const esperarBoton = new MutationObserver(() => {
  if (document.querySelector('.modo')) { esperarBoton.disconnect(); setTimeout(aplicarModoGuardado, 0); }
});
esperarBoton.observe(document, {subtree: true, childList: true});
document.addEventListener('click', e => {
  if (!e.target.closest('.modo')) return;
  const oscuro = document.body.classList.toggle('dark');
  try { localStorage.setItem('modo', oscuro ? 'oscuro' : 'claro'); } catch (e) {}
  rotularModo();
});
</script>
"""

CSS = """
:root {
  --papel: #F6F7F4; --tinta: #1C2733; --tinta-suave: #55606B; --regla: #D9DDD6;
  --vigente: #22614F; --derogada: #8A2D3B; --pregunta: #E7EBEE; --campo: #FFFFFF; --campo-borde: #C9CEC6;
}
/* Modo oscuro: la tinta pasa a ser la página. Mismos roles, contraste AA verificado. */
body.dark {
  --papel: #17212B; --tinta: #E3E6E0; --tinta-suave: #A1ABB4; --regla: #33414E;
  --vigente: #7FC4A7; --derogada: #E8959F; --pregunta: #223040; --campo: #1D2833; --campo-borde: #3D4B58;
}
footer { display: none !important; }

.cabecera { max-width: 1180px; margin: 0 auto; padding: 28px 0 18px; border-bottom: 1px solid var(--regla);
            display: flex; justify-content: space-between; align-items: flex-end; gap: 16px; }
.cabecera h1 { font-family: "Newsreader", Georgia, serif; font-weight: 500; font-size: 2rem; line-height: 1.15;
               letter-spacing: -0.01em; color: var(--tinta); margin: 0 0 6px; }
.cabecera p { color: var(--tinta-suave); font-size: 1rem; margin: 0; }
.modo { font: inherit; font-size: 0.9rem; color: var(--tinta-suave); background: transparent; cursor: pointer;
        border: 1px solid var(--regla); border-radius: 4px; padding: 6px 12px; white-space: nowrap; }
.modo:hover { color: var(--tinta); border-color: var(--tinta-suave); }
.aviso { text-align: center; color: var(--tinta-suave); font-size: 0.85rem; margin: 6px 0 0; }

/* Selector de modelo: una línea discreta, no un formulario */
.form:has(> #modelo) { background: transparent !important; border: none !important; box-shadow: none !important;
                       max-width: 1180px; width: 100%; margin: 0 auto; }
#modelo { max-width: 1180px; margin: 0 auto; padding: 0 !important; background: transparent !important; border: none !important; }
#modelo > span, #modelo [data-testid="block-info"] { font-size: 0.9rem !important; color: var(--tinta-suave) !important; margin-bottom: 6px !important; }
#modelo .wrap { gap: 0 !important; }
#modelo label { background: transparent !important; border: 1px solid var(--regla) !important; box-shadow: none !important;
                border-radius: 0 !important; margin: 0 0 0 -1px !important; padding: 6px 12px !important; color: var(--tinta-suave) !important;
                gap: 0 !important; cursor: pointer; }
#modelo label:first-child { border-radius: 4px 0 0 4px !important; margin-left: 0 !important; }
#modelo label:last-child { border-radius: 0 4px 4px 0 !important; }
#modelo label.selected { color: var(--tinta) !important; border-color: var(--tinta) !important; position: relative; z-index: 1; }
#modelo label input { display: none !important; }

/* Estado vacío */
.vacio { text-align: left; max-width: 34rem; }
.vacio-t { font-family: "Newsreader", Georgia, serif; font-size: 1.6rem; font-weight: 500; margin: 0 0 6px; color: var(--tinta); }
.vacio p.vacio-d { color: var(--tinta-suave); margin: 0 0 4px; }
.vacio p.vacio-d + p.vacio-d { margin-top: 18px; }
.sugerencias { list-style: none; padding: 0 !important; margin: 6px 0 0 !important; }
.sugerencias li { margin: 0 !important; padding: 0 !important; }
.sugerencias li::marker { content: ""; }
.sugerencia { display: inline-block; padding: 5px 0; color: var(--tinta) !important; text-decoration: underline;
              text-decoration-color: var(--regla); text-underline-offset: 4px; }
.sugerencia:hover { text-decoration-color: var(--tinta); }

/* Lista de normas */
.buscando { color: var(--tinta-suave); margin: 0 !important; }
.normas { margin-top: 18px; padding-top: 12px; border-top: 1px solid var(--regla); }
.normas.sin-normas { border-top: none; padding-top: 0; }
.normas-titulo { font-size: 0.9rem; color: var(--tinta-suave); margin: 0 0 6px !important; }
.normas ul { list-style: none; padding: 0 !important; margin: 0 !important; }
.normas li.norma { display: flex; gap: 14px; align-items: baseline; padding: 5px 0 5px 12px !important;
                   margin: 0 !important; border-left: 3px solid var(--vigente); }
.normas li.norma + li.norma { margin-top: 4px !important; }
.normas li.norma::marker { content: ""; }
.normas .nombre, .normas del { font-family: "Newsreader", Georgia, serif; font-size: 1.05rem; color: var(--tinta); }
.normas .estado { font-size: 0.9rem; color: var(--vigente); }
.normas li.norma-derogada { border-left-color: var(--derogada); }
.normas li.norma-derogada del { text-decoration-color: var(--derogada); text-decoration-thickness: 1.5px; color: var(--tinta-suave); }
.normas li.norma-derogada .estado { color: var(--derogada); }
.normas li.norma-neutra { border-left-color: var(--regla); }
.normas li.norma-neutra .estado { color: var(--tinta-suave); }
.modelo-usado { font-size: 0.85rem; color: var(--tinta-suave); margin: 10px 0 0 !important; }

/* Columna de lectura: la respuesta se lee como un texto, no como un chat de soporte */
#chat .bubble-wrap { background: transparent !important; }
#chat .message-wrap { max-width: 680px; margin: 0 auto; }
#chat .message-row.bot-row .message { background: transparent !important; border: none !important; box-shadow: none !important; padding-left: 0 !important; }
#chat .message-row.bot-row .prose { font-size: 1.02rem; line-height: 1.65; color: var(--tinta); }
#chat .message-row.user-row .message { background: var(--pregunta) !important; border: none !important; border-radius: 6px !important; }
#chat .message-row.user-row .prose { color: var(--tinta); }
#chat .prose strong { font-weight: 600; }
#chat .vacio, #chat .vacio * { opacity: 1 !important; }
#chat button[aria-label="Scroll down"] { background: var(--papel) !important; border: 1px solid var(--regla) !important;
                                         color: var(--tinta) !important; border-radius: 50% !important; }

/* Barra lateral: historial de consultas */
.column:has(> button):has(table) { gap: 14px !important; }
.column:has(> button):has(table) > button { white-space: nowrap; justify-content: flex-start; padding-left: 14px !important; }
.column:has(> button):has(table) .block { border: none !important; background: transparent !important; padding: 0 !important; }
.column:has(> button):has(table) table, .column:has(> button):has(table) .table-wrap { border: none !important; background: transparent !important; }
.column:has(> button):has(table) tr { background: transparent !important; }
.column:has(> button):has(table) td { text-align: left !important; padding: 7px 10px !important; border: none !important;
                                       background: transparent !important; color: var(--tinta-suave); font-size: 0.92rem;
                                       border-left: 2px solid transparent !important; }
.column:has(> button):has(table) tr:hover td { color: var(--tinta); border-left-color: var(--regla) !important; }
.column:has(> button):has(table) tr.selected td, .column:has(> button):has(table) tr[aria-selected="true"] td { color: var(--tinta); border-left-color: var(--tinta) !important; }

/* Entrada */
#entrada { border: 1px solid var(--campo-borde) !important; border-radius: 6px !important; background: var(--campo) !important; padding: 4px !important; }
#entrada:focus-within { border-color: var(--tinta) !important; }
#entrada textarea { font-size: 1rem; background: transparent !important; color: var(--tinta) !important; }
#entrada textarea::placeholder { color: var(--tinta-suave); }
#entrada .submit-button { background: var(--tinta) !important; color: var(--papel) !important; border-radius: 4px !important; padding: 0 18px !important; }
:focus-visible { outline: 2px solid var(--tinta) !important; outline-offset: 2px; }

@media (max-width: 720px) {
  .cabecera { padding: 18px 0 12px; flex-direction: column; align-items: flex-start; gap: 12px; }
  .cabecera h1 { font-size: 1.5rem; }
  .normas li.norma { flex-direction: column; gap: 0; }
  #modelo .wrap { flex-direction: column; align-items: stretch; gap: 6px !important; }
  #modelo label, #modelo label:first-child, #modelo label:last-child { margin: 0 !important; border-radius: 4px !important; }
}
"""

tema = gr.themes.Base(
    font=["Public Sans", "system-ui", "sans-serif"],
    font_mono=["ui-monospace", "monospace"],
    radius_size=gr.themes.sizes.radius_sm,
).set(
    body_background_fill="#F6F7F4", body_text_color="#1C2733",
    block_background_fill="#F6F7F4", block_border_width="0px", block_shadow="none",
    button_primary_background_fill="#1C2733", button_primary_background_fill_hover="#2C3B4A",
    button_primary_text_color="#F6F7F4", input_background_fill="#FFFFFF",
    input_border_color="#C9CEC6", input_border_color_focus="#1C2733",
    body_background_fill_dark="#17212B", body_text_color_dark="#E3E6E0",
    body_text_color_subdued_dark="#A1ABB4",
    background_fill_primary_dark="#17212B", background_fill_secondary_dark="#1D2833",
    border_color_primary_dark="#33414E",
    block_background_fill_dark="#17212B", block_border_width_dark="0px",
    button_primary_background_fill_dark="#E3E6E0", button_primary_background_fill_hover_dark="#C9CEC6",
    button_primary_text_color_dark="#17212B",
    button_secondary_background_fill_dark="#1D2833", button_secondary_text_color_dark="#E3E6E0",
    input_background_fill_dark="#1D2833", input_border_color_dark="#3D4B58",
    input_border_color_focus_dark="#E3E6E0",
)

VACIO = ('<div class="vacio"><p class="vacio-t">¿Qué querés saber?</p>'
         '<p class="vacio-d">Preguntá por un régimen, un incentivo o si una ley sigue vigente.</p>'
         '<p class="vacio-d">Por ejemplo:</p><ul class="sugerencias">'
         + "".join(f'<li><a href="#" class="sugerencia">{html.escape(q)}</a></li>' for q in SUGERENCIAS)
         + '</ul></div>')

with gr.Blocks(title="Leyes de inversión del Paraguay") as demo:
    gr.HTML('<header class="cabecera"><div><h1>Leyes de inversión del Paraguay</h1>'
            '<p>Cada respuesta cita la norma en la que se basa y avisa si fue derogada.</p></div>'
            '<button type="button" class="modo">Modo oscuro</button></header>')
    selector = gr.Radio(list(MODELOS), value=next(iter(MODELOS)), label="Modelo que responde",
                        elem_id="modelo", visible=MOSTRAR_SELECTOR)
    gr.ChatInterface(
        fn=responder_ui,
        additional_inputs=[selector],
        chatbot=gr.Chatbot(elem_id="chat", show_label=False, height=620, placeholder=VACIO),
        textbox=gr.Textbox(placeholder="Escribí tu consulta", submit_btn="Enviar", elem_id="entrada"),
        save_history=True, fill_height=True,
    )
    gr.HTML('<p class="aviso">Prototipo académico. No constituye asesoramiento legal.</p>')

demo.queue(default_concurrency_limit=1)        # una consulta a la vez: hay una sola GPU
demo.launch(share=True, theme=tema, css=CSS, head=HEAD)